In [1]:
import pandas as pd

usd_vnd_raw = pd.read_excel(
    "d:/HK3 - năm 3/Đồ Án/Code/vietnam-transportation-cpi-forecasting/data/raw/usd_vnd_exchange_rate.xlsx",
    sheet_name="Tỷ giá USDVND"
)

usd_vnd_raw.head()

,Chỉ tiêu,03-01-2009,05-01-2009,06-01-2009,07-01-2009,08-01-2009,09-01-2009,10-01-2009,12-01-2009,13-01-2009,...,25-07-2026,27-07-2026,28-07-2026,29-07-2026,30-07-2026,31-07-2026,03-08-2026,04-08-2026,05-08-2026,06-08-2026
0,Trung tâm,16973.0,16973.0,16971.0,16972.0,16970.0,16973.0,16974.0,16974.0,16976.0,...,25283.0,25293,25306,25306,25320,25338,NaN,NaN,NaN,NaN
1,Tỷ giá trần,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,26547.0,26558,26571,26571,26586,26605,NaN,NaN,NaN,NaN
2,Tỷ giá sàn,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,24019.0,24028,24041,24041,24054,24071,NaN,NaN,NaN,NaN
3,Mua vào,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,24079,24091,24091,24104,24122,24141.0,24161.0,24185.0,24212.0
4,Bán ra,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,26507,26521,26521,26536,26554,26575.0,26599.0,26625.0,26654.0


## Lựa chọn tỷ giá USD/VND

Đề tài sử dụng tỷ giá trung tâm USD/VND làm biến đại diện cho biến động tỷ giá. Dữ liệu tỷ giá trung tâm được trích xuất từ bảng dữ liệu gốc trước khi chuẩn hóa về tần suất tháng.

In [2]:
usd_vnd = usd_vnd_raw[
    usd_vnd_raw["Chỉ tiêu"] == "Trung tâm"
].copy()
usd_vnd

,Chỉ tiêu,03-01-2009,05-01-2009,06-01-2009,07-01-2009,08-01-2009,09-01-2009,10-01-2009,12-01-2009,13-01-2009,...,25-07-2026,27-07-2026,28-07-2026,29-07-2026,30-07-2026,31-07-2026,03-08-2026,04-08-2026,05-08-2026,06-08-2026
0,Trung tâm,16973.0,16973.0,16971.0,16972.0,16970.0,16973.0,16974.0,16974.0,16976.0,...,25283.0,25293,25306,25306,25320,25338,NaN,NaN,NaN,NaN


In [3]:
usd_vnd_long = usd_vnd.melt(
    id_vars="Chỉ tiêu",
    var_name="Date",
    value_name="USD_VND"
).drop(columns=["Chỉ tiêu"])
usd_vnd_long.head()

,Date,USD_VND
0,03-01-2009,16973.0
1,05-01-2009,16973.0
2,06-01-2009,16971.0
3,07-01-2009,16972.0
4,08-01-2009,16970.0


In [4]:
usd_vnd_long["Date"] = pd.to_datetime(
    usd_vnd_long["Date"],
    dayfirst=True
)
usd_vnd_long.dtypes

Date       datetime64[us]
USD_VND           float64
dtype: object

In [5]:
usd_vnd_long = usd_vnd_long[
    (usd_vnd_long["Date"] >= "2011-09-01") &
    (usd_vnd_long["Date"] <= "2024-12-31")
].copy()
print(
    usd_vnd_long["Date"].min(),
    "->",
    usd_vnd_long["Date"].max()
)

2011-09-01 00:00:00 -> 2024-12-31 00:00:00


In [6]:
usd_vnd_long["USD_VND"].isna().sum()

np.int64(135)

In [7]:
usd_vnd_long[
    usd_vnd_long["USD_VND"].isna()
].groupby(
    usd_vnd_long["Date"].dt.year
).size()

Date
2022    52
2023    41
2024    42
dtype: int64

In [8]:
usd_vnd_long.loc[
    usd_vnd_long["USD_VND"].isna(),
    "Date"
].dt.day_name().value_counts()

Date
Tuesday      33
Wednesday    32
Monday       31
Thursday     23
Friday       16
Name: count, dtype: int64

In [9]:
usd_vnd_long["MonthYear"] = usd_vnd_long["Date"].dt.to_period("M")

usd_vnd_long.groupby("MonthYear")["USD_VND"].count().sort_values().head()

MonthYear
2023-01    15
2021-02    16
2019-02    17
2015-02    17
2021-05    17
Freq: M, Name: USD_VND, dtype: int64

In [10]:
usd_vnd_monthly = (
    usd_vnd_long
    .groupby("MonthYear")["USD_VND"]
    .mean()
    .round(2)
    .reset_index()
)
usd_vnd_monthly.head(20)

,MonthYear,USD_VND
0,2011-09,20628.0
1,2011-10,20713.0
2,2011-11,20803.0
3,2011-12,20813.6
4,2012-01,20828.0
5,2012-02,20828.0
6,2012-03,20828.0
7,2012-04,20828.0
8,2012-05,20828.0
9,2012-06,20828.0


In [11]:
print(usd_vnd_monthly.shape)

(160, 2)


In [12]:
usd_vnd_monthly.isna().sum()

MonthYear    0
USD_VND      0
dtype: int64

In [13]:
usd_vnd_monthly[usd_vnd_monthly["USD_VND"]<0]

,MonthYear,USD_VND


In [14]:
usd_vnd_monthly.to_csv(
    "d:/HK3 - năm 3/Đồ Án/Code/vietnam-transportation-cpi-forecasting/data/interim/usd_vnd_monthly.csv",
    index=False,
    encoding="utf-8-sig"
)